In [3]:
from pathlib import Path

BASE = Path(r"E:\AI_Detect")

folders = [
    "data/text/human/raw",
    "data/text/human/cleaned",
    "data/text/human/metadata",

    "data/text/ai/raw",
    "data/text/ai/cleaned",
    "data/text/ai/metadata",

    "data/image",
    "data/audio",
    "data/video",
    "data/multimodal",

    "generators",
    "scripts",
    "logs",
    "benchmark_splits",
    "database"
]

for f in folders:
    p = BASE / f
    p.mkdir(parents=True, exist_ok=True)

print("Folders created.")

Folders created.


In [1]:
import requests
import pandas as pd

url = "https://archive.org/advancedsearch.php"

params = {
    "q": "mediatype:texts AND year:[2000 TO 2015]",
    "fl[]": ["identifier", "title", "year"],
    "rows": 20,
    "page": 1,
    "output": "json"
}

r = requests.get(url, params=params)

data = r.json()

docs = data["response"]["docs"]

df = pd.DataFrame(docs)

df.head()

,identifier,title,year
0,Temporary_Inspection_Tag_VS_1077,Temporary_Inspection_Tag_VS_1077,2008
1,cheminsdailleurs0000trot,Chemins d'ailleurs : chroniques et anecdotes d...,2011
2,VideoCast.com,VideoCast,2005
3,juveniledelinque0000unse_k3z4,Juvenile delinquency : a justice perspective,2000
4,godwholoves0000maca,The God who loves,2001


In [4]:
save_path = BASE / "data/text/human/metadata/archive_text_metadata.csv"

df.to_csv(save_path, index=False)

print("Saved:", save_path)

Saved: E:\AI_Detect\data\text\human\metadata\archive_text_metadata.csv


In [5]:
import requests
import pandas as pd
from tqdm import tqdm
from pathlib import Path

BASE = Path(r"E:\AI_Detect")

save_csv = BASE / "database/archive_text_candidates.csv"

all_docs = []

rows_per_page = 100

pages = 50   # increase later

for page in tqdm(range(1, pages + 1)):

    params = {
        "q": "mediatype:texts AND year:[2000 TO 2018]",
        "fl[]": [
            "identifier",
            "title",
            "year",
            "language",
            "mediatype"
        ],
        "rows": rows_per_page,
        "page": page,
        "output": "json"
    }

    url = "https://archive.org/advancedsearch.php"

    try:

        r = requests.get(url, params=params, timeout=30)

        docs = r.json()["response"]["docs"]

        all_docs.extend(docs)

    except Exception as e:
        print("ERROR PAGE:", page, e)

df = pd.DataFrame(all_docs)

df.to_csv(save_csv, index=False)

print("Total collected:", len(df))
print("Saved to:", save_csv)

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:43<00:00,  1.14it/s]

Total collected: 5000
Saved to: E:\AI_Detect\database\archive_text_candidates.csv


In [6]:
import pandas as pd

csv_path = BASE / "database/archive_text_candidates.csv"

df = pd.read_csv(csv_path)

df = df.drop_duplicates(subset=["identifier"])

df = df[
    (
        df["language"].astype(str)
        .str.contains("eng", case=False, na=False)
    )
]

print("English items:", len(df))

df.head()

English items: 3911


,identifier,language,mediatype,title,year
4,jugglingtales0000taig,eng,texts,Juggling tales,2011
5,justfactsabortio0000moor,eng,texts,Just the Facts: Abortion A to Z,2007
7,karenbrownsnewen00kare,eng,texts,Karen Brown's New England : charming inns & it...,2004
8,sheppardlee0000robe_c3j7,eng,texts,Sheppard Lee,2004
10,sogreatcloudreco0000redm,eng,texts,So great a cloud : a record of Christian witness,2009


In [7]:
filtered_path = BASE / "database/archive_text_english.csv"

df.to_csv(filtered_path, index=False)

print(filtered_path)

E:\AI_Detect\database\archive_text_english.csv


In [8]:
import requests
from pathlib import Path
from tqdm import tqdm
import time

raw_dir = BASE / "data/text/human/raw"

sample_df = df.head(5000)

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):

    identifier = row["identifier"]

    save_file = raw_dir / f"{identifier}.txt"

    if save_file.exists():
        continue

    try:

        meta_url = f"https://archive.org/metadata/{identifier}"

        meta = requests.get(meta_url, timeout=20).json()

        files = meta.get("files", [])

        txt_files = [
            f["name"]
            for f in files
            if f["name"].endswith(".txt")
        ]

        if len(txt_files) == 0:
            continue

        txt_name = txt_files[0]

        download_url = (
            f"https://archive.org/download/"
            f"{identifier}/{txt_name}"
        )

        r = requests.get(download_url, timeout=30)

        if r.status_code != 200:
            continue

        with open(save_file, "wb") as f:
            f.write(r.content)

        time.sleep(0.2)

    except Exception as e:
        print("ERROR:", identifier, e)

 40%|██████████████████████████████                                              | 1547/3911 [57:43<4:55:09,  7.49s/it]

ERROR: pastchances0000kenn ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


 87%|████████████████████████████████████████████████████████████████▏         | 3391/3911 [2:06:01<1:11:09,  8.21s/it]

ERROR: texesprincipal060000shar HTTPSConnectionPool(host='dn720507.ca.archive.org', port=443): Max retries exceeded with url: /0/items/texesprincipal060000shar/texesprincipal060000shar_djvu.txt (Caused by ConnectTimeoutError(<HTTPSConnection(host='dn720507.ca.archive.org', port=443) at 0x17867419030>, 'Connection to dn720507.ca.archive.org timed out. (connect timeout=30)'))


 87%|████████████████████████████████████████████████████████████████▏         | 3392/3911 [2:06:23<1:46:59, 12.37s/it]

...

 97%|█████████████████████████████████████████████████████████████████████████▉  | 3802/3911 [2:23:34<13:32,  7.46s/it]

ERROR: goodshoppingguid0000unse_p8n5 HTTPSConnectionPool(host='archive.org', port=443): Read timed out. (read timeout=20)


 99%|███████████████████████████████████████████████████████████████████████████ | 3865/3911 [2:26:23<06:10,  8.06s/it]

ERROR: lookingatworldtw0000mann HTTPSConnectionPool(host='archive.org', port=443): Max retries exceeded with url: /download/lookingatworldtw0000mann/lookingatworldtw0000mann_djvu.txt (Caused by ConnectTimeoutError(<HTTPSConnection(host='archive.org', port=443) at 0x1786741bd30>, 'Connection to archive.org timed out. (connect timeout=30)'))


100%|████████████████████████████████████████████████████████████████████████████| 3911/3911 [2:28:10<00:00,  2.27s/it]


In [9]:
from pathlib import Path

raw_dir = BASE / "data/text/human/raw"

files = list(raw_dir.glob("*.txt"))

print("Downloaded:", len(files))

Downloaded: 1207


In [10]:
import re
from pathlib import Path
from ftfy import fix_text

clean_dir = BASE / "data/text/human/cleaned"

files = list(raw_dir.glob("*.txt"))

for fp in files:

    try:

        text = fp.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        text = fix_text(text)

        text = re.sub(r"\s+", " ", text)

        text = text.strip()

        if len(text) < 1000:
            continue

        out_file = clean_dir / fp.name

        out_file.write_text(text, encoding="utf-8")

    except Exception as e:
        print(e)

In [6]:
# ============================================================
# LARGE-SCALE HUMAN TEXT COLLECTION PIPELINE
# Sources:
#   1. Wikipedia
#   2. ArXiv
#   3. Reddit (Pushshift)
#
# Saves EVERYTHING into:
#   E:\AI_Detect\data\text\human\cleaned
#
# Includes:
#   - tqdm progress bars
#   - cleaning
#   - deduplication
#   - metadata csv
#   - resumable downloading
# ============================================================

!pip install wikipedia-api arxiv praw psaw ftfy tqdm pandas requests -q

import os
import re
import time
import hashlib
import random
import requests
import pandas as pd

from pathlib import Path
from tqdm import tqdm
from ftfy import fix_text

import wikipediaapi
import arxiv

# ============================================================
# PATHS
# ============================================================

BASE = Path(r"E:\AI_Detect")

CLEAN_DIR = BASE / "data/text/human/cleaned"

META_DIR = BASE / "data/text/human/metadata"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

META_FILE = META_DIR / "human_text_metadata.csv"

# ============================================================
# CONFIG
# ============================================================

TARGET_WIKI = 5000
TARGET_ARXIV = 5000
TARGET_REDDIT = 10000

MIN_WORDS = 100

# ============================================================
# DEDUP TRACKER
# ============================================================

seen_hashes = set()

existing_files = list(CLEAN_DIR.glob("*.txt"))

for fp in existing_files:

    try:

        txt = fp.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        h = hashlib.md5(txt.encode()).hexdigest()

        seen_hashes.add(h)

    except:
        pass

print("Existing hashes loaded:", len(seen_hashes))

# ============================================================
# CLEANER
# ============================================================

def clean_text(text):

    text = fix_text(text)

    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text

# ============================================================
# SAVE FUNCTION
# ============================================================

metadata_rows = []

def save_text(text, source, extra={}):

    global seen_hashes

    text = clean_text(text)

    words = text.split()

    if len(words) < MIN_WORDS:
        return False

    h = hashlib.md5(text.encode()).hexdigest()

    if h in seen_hashes:
        return False

    seen_hashes.add(h)

    fname = f"{source}_{h}.txt"

    save_path = CLEAN_DIR / fname

    save_path.write_text(
        text,
        encoding="utf-8"
    )

    row = {

        "filename": fname,
        "source": source,
        "words": len(words),
        "chars": len(text)

    }

    row.update(extra)

    metadata_rows.append(row)

    return True

# ============================================================
# WIKIPEDIA COLLECTION
# ============================================================

print("\n================ WIKIPEDIA =================")

wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='AI_Detect_Benchmark/1.0'
)

topics = [

    "Science",
    "History",
    "Physics",
    "Biology",
    "Computer science",
    "Mathematics",
    "Medicine",
    "Astronomy",
    "Engineering",
    "Artificial intelligence"

]

wiki_saved = 0

for topic in topics:

    if wiki_saved >= TARGET_WIKI:
        break

    try:

        page = wiki.page(topic)

        links = list(page.links.keys())

        random.shuffle(links)

        for title in tqdm(
            links,
            desc=f"Wikipedia-{topic}"
        ):

            if wiki_saved >= TARGET_WIKI:
                break

            try:

                p = wiki.page(title)

                text = p.text

                ok = save_text(

                    text=text,
                    source="wiki",

                    extra={

                        "topic": topic,
                        "title": title

                    }

                )

                if ok:
                    wiki_saved += 1

            except:
                continue

    except:
        pass

print("Wikipedia saved:", wiki_saved)

# ============================================================
# ARXIV COLLECTION
# ============================================================

print("\n================ ARXIV =================")

search = arxiv.Search(

    query="cat:cs.AI OR cat:cs.LG",

    max_results=TARGET_ARXIV * 2,

    sort_by=arxiv.SortCriterion.SubmittedDate

)

arxiv_saved = 0

for result in tqdm(

    search.results(),
    desc="ArXiv"

):

    if arxiv_saved >= TARGET_ARXIV:
        break

    try:

        year = result.published.year

        if year > 2020:
            continue

        text = ""

        text += result.title + "\n\n"

        text += result.summary + "\n\n"

        authors = ", ".join(
            [a.name for a in result.authors]
        )

        text += authors

        ok = save_text(

            text=text,
            source="arxiv",

            extra={

                "year": year,
                "title": result.title

            }

        )

        if ok:
            arxiv_saved += 1

    except:
        continue

print("ArXiv saved:", arxiv_saved)

# ============================================================
# REDDIT COLLECTION
# ============================================================

print("\n================ REDDIT =================")

subreddits = [

    "AskReddit",
    "technology",
    "science",
    "worldnews",
    "todayilearned",
    "explainlikeimfive",
    "programming"

]

reddit_saved = 0

for sub in subreddits:

    if reddit_saved >= TARGET_REDDIT:
        break

    print("\nSubreddit:", sub)

    url = (
        "https://api.pushshift.io/"
        "reddit/search/comment/"
    )

    params = {

        "subreddit": sub,

        "size": 1000,

        "before": "2020-01-01"

    }

    try:

        r = requests.get(
            url,
            params=params,
            timeout=30
        )

        data = r.json()["data"]

        for item in tqdm(
            data,
            desc=f"Reddit-{sub}"
        ):

            if reddit_saved >= TARGET_REDDIT:
                break

            body = item.get("body", "")

            ok = save_text(

                text=body,
                source="reddit",

                extra={

                    "subreddit": sub,
                    "year": 2019

                }

            )

            if ok:
                reddit_saved += 1

            time.sleep(0.005)

    except Exception as e:

        print("ERROR:", sub, e)

print("Reddit saved:", reddit_saved)

# ============================================================
# SAVE METADATA
# ============================================================

meta_df = pd.DataFrame(metadata_rows)

if META_FILE.exists():

    old_df = pd.read_csv(META_FILE)

    meta_df = pd.concat(
        [old_df, meta_df],
        ignore_index=True
    )

meta_df.to_csv(
    META_FILE,
    index=False
)

# ============================================================
# FINAL STATS
# ============================================================

files = list(CLEAN_DIR.glob("*.txt"))

total_words = 0

for fp in tqdm(
    files,
    desc="Final Stats"
):

    try:

        txt = fp.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        total_words += len(txt.split())

    except:
        pass

print("\n================================================")
print("FINAL DATASET STATS")
print("================================================")

print("Total files:", len(files))

print("Total words:", f"{total_words:,}")

print("Wikipedia added:", wiki_saved)

print("ArXiv added:", arxiv_saved)

print("Reddit added:", reddit_saved)

print("\nSaved in:")
print(CLEAN_DIR)

print("\nMetadata:")
print(META_FILE)

Existing hashes loaded: 963

================ WIKIPEDIA =================


Wikipedia-Astronomy:  62%|████████████████████████████████████▋                      | 495/797 [01:40<01:01,  4.93it/s]


Wikipedia saved: 5000

================ ARXIV =================


AttributeError: 'Search' object has no attribute 'results'

In [7]:
!pip install arxiv --upgrade

import arxiv
import os

from tqdm import tqdm

# =====================================
# SAVE PATH
# =====================================

SAVE_DIR = r"E:\Pred_Market\arxiv_raw"

os.makedirs(SAVE_DIR, exist_ok=True)

# =====================================
# SETTINGS
# =====================================

TARGET_ARXIV = 5000

# =====================================
# CLIENT
# =====================================

client = arxiv.Client()

search = arxiv.Search(

    query="cat:cs.AI OR cat:cs.LG",

    max_results=TARGET_ARXIV,

    sort_by=arxiv.SortCriterion.SubmittedDate

)

# =====================================
# DOWNLOAD
# =====================================

saved = 0

for result in tqdm(

    client.results(search),

    total=TARGET_ARXIV,
    desc="ArXiv"

):

    try:

        text = ""

        text += "TITLE:\n"
        text += result.title + "\n\n"

        text += "ABSTRACT:\n"
        text += result.summary + "\n\n"

        text += "CATEGORIES:\n"
        text += str(result.categories) + "\n\n"

        text += "PUBLISHED:\n"
        text += str(result.published) + "\n\n"

        # save
        out_file = os.path.join(
            SAVE_DIR,
            f"{saved}.txt"
        )

        with open(
            out_file,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(text)

        saved += 1

    except Exception as e:

        print("Error:", e)

print("\nSaved:", saved)

ArXiv: 100%|███████████████████████████████████████████████████████████████████████| 5000/5000 [03:05<00:00, 26.96it/s]


Saved: 5000


In [8]:
!pip install requests pandas tqdm ftfy praw

In [13]:
# ============================================================
# WORKING REDDIT HUMAN DATA COLLECTOR
# USING MODERN HUGGINGFACE PARQUET DATASETS
# ============================================================

!pip install datasets pandas tqdm ftfy -q

import re
import hashlib
import pandas as pd

from pathlib import Path
from tqdm import tqdm
from ftfy import fix_text

from datasets import load_dataset

# ============================================================
# PATHS
# ============================================================

BASE = Path(r"E:\AI_Detect")

CLEAN_DIR = BASE / "data/text/human/cleaned"

META_DIR = BASE / "data/text/human/metadata"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

META_FILE = META_DIR / "reddit_metadata.csv"

# ============================================================
# CONFIG
# ============================================================

TARGET = 50000

MIN_WORDS = 20
MAX_WORDS = 1000

# ============================================================
# LOAD EXISTING HASHES
# ============================================================

seen_hashes = set()

existing = list(
    CLEAN_DIR.glob("reddit_*.txt")
)

for fp in existing:

    try:

        txt = fp.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        h = hashlib.md5(
            txt.encode()
        ).hexdigest()

        seen_hashes.add(h)

    except:
        pass

print("Existing reddit files:", len(existing))

# ============================================================
# CLEANER
# ============================================================

def clean_text(text):

    text = fix_text(text)

    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text

# ============================================================
# QUALITY FILTER
# ============================================================

bad_phrases = [

    "[deleted]",
    "[removed]",
    "i am a bot",
    "automoderator"

]

def valid_comment(text):

    t = text.lower()

    for b in bad_phrases:

        if b in t:
            return False

    words = text.split()

    if len(words) < MIN_WORDS:
        return False

    if len(words) > MAX_WORDS:
        return False

    return True

# ============================================================
# LOAD WORKING DATASET
# ============================================================

print("\nLoading Reddit dataset...")

dataset = load_dataset(

    "HuggingFaceFW/fineweb",

    split="train",

    streaming=True

)

print("Dataset stream ready.")

# ============================================================
# COLLECTION
# ============================================================

metadata_rows = []

saved = 0

pbar = tqdm(total=TARGET)

for item in dataset:

    try:

        # fineweb text field
        text = item["text"]

        # keep only Reddit-like URLs if present
        if "reddit.com" not in item.get("url", ""):
            continue

        text = clean_text(text)

        if not valid_comment(text):
            continue

        # dedup
        h = hashlib.md5(
            text.encode()
        ).hexdigest()

        if h in seen_hashes:
            continue

        seen_hashes.add(h)

        # save
        fname = f"reddit_{h}.txt"

        save_path = CLEAN_DIR / fname

        save_path.write_text(

            text,

            encoding="utf-8"

        )

        metadata_rows.append({

            "filename": fname,
            "source": "reddit",
            "words": len(text.split()),
            "chars": len(text)

        })

        saved += 1

        pbar.update(1)

        if saved >= TARGET:
            break

    except Exception as e:

        print("ERROR:", e)

pbar.close()

# ============================================================
# SAVE METADATA
# ============================================================

meta_df = pd.DataFrame(metadata_rows)

if META_FILE.exists():

    old_df = pd.read_csv(META_FILE)

    meta_df = pd.concat(

        [old_df, meta_df],

        ignore_index=True

    )

meta_df.to_csv(

    META_FILE,

    index=False

)

# ============================================================
# FINAL STATS
# ============================================================

files = list(
    CLEAN_DIR.glob("reddit_*.txt")
)

total_words = 0

for fp in tqdm(

    files,

    desc="Counting words"

):

    try:

        txt = fp.read_text(

            encoding="utf-8",

            errors="ignore"

        )

        total_words += len(txt.split())

    except:
        pass

print("\n================================================")
print("REDDIT DATASET COMPLETE")
print("================================================")

print("Total reddit files:", len(files))

print("Total words:", f"{total_words:,}")

print("Metadata saved:", META_FILE)

Existing reddit files: 2731

Loading Reddit dataset...
Dataset stream ready.



  5%|███▉                                                                    | 2731/50000 [1:24:43<24:26:31,  1.86s/it]

  2%|█▏                                                                       | 804/50000 [1:31:11<14:00:25,  1.03s/it]'[Errno 11001] getaddrinfo failed' thrown while requesting GET https://huggingface.co/datasets/HuggingFaceFW/fineweb/resolve/9bb295ddab0e05d785b879661af7260fed5140fc/data/CC-MAIN-2013-20/000_00017.parquet
Retrying in 1s [Retry 1/5].

  2%|█▏                                                                       | 817/50000 [1:31:34<23:33:28,  1.72s/it]

RuntimeError: Cannot send a request, as the client has been closed.

In [1]:
from pathlib import Path

BASE = Path(r"E:\AI_Detect")

def folder_stats(folder):

    total_size = 0
    total_files = 0

    for fp in folder.rglob("*"):

        if fp.is_file():

            total_size += fp.stat().st_size
            total_files += 1

    return total_files, total_size

for folder in BASE.iterdir():

    if folder.is_dir():

        n, s = folder_stats(folder)

        gb = s / (1024**3)

        print(
            f"{folder.name:20s} | "
            f"Files: {n:8d} | "
            f"Size: {gb:.4f} GB"
        )

.git                 | Files:       28 | Size: 0.0000 GB
benchmark_splits     | Files:        0 | Size: 0.0000 GB
data                 | Files:     2174 | Size: 0.0788 GB
database             | Files:        2 | Size: 0.0008 GB
generators           | Files:        0 | Size: 0.0000 GB
logs                 | Files:        0 | Size: 0.0000 GB
scripts              | Files:        0 | Size: 0.0000 GB


In [1]:
from pathlib import Path

clean_dir = Path(r"E:\AI_Detect\data\text\human\cleaned")

files = list(clean_dir.glob("*.txt"))

total_words = 0
total_chars = 0

largest_words = 0
smallest_words = 10**18

largest_file = None
smallest_file = None

for fp in files:

    try:

        text = fp.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        words = text.split()

        n_words = len(words)
        n_chars = len(text)

        total_words += n_words
        total_chars += n_chars

        if n_words > largest_words:

            largest_words = n_words
            largest_file = fp.name

        if n_words < smallest_words:

            smallest_words = n_words
            smallest_file = fp.name

    except Exception as e:

        print("ERROR:", fp.name, e)

num_files = len(files)

avg_words = total_words / max(num_files, 1)

print("=" * 50)

print("CLEANED TEXT DATASET STATS")

print("=" * 50)

print(f"Total files          : {num_files}")
print(f"Total words          : {total_words:,}")
print(f"Total characters     : {total_chars:,}")
print(f"Average words/file   : {avg_words:,.2f}")

print()

print(f"Largest file         : {largest_file}")
print(f"Largest word count   : {largest_words:,}")

print()

print(f"Smallest file        : {smallest_file}")
print(f"Smallest word count  : {smallest_words:,}")

CLEANED TEXT DATASET STATS
Total files          : 14514
Total words          : 27,739,058
Total characters     : 177,834,226
Average words/file   : 1,911.19

Largest file         : etsi_ts_125_331_v07.12.01.txt
Largest word count   : 487,805

Smallest file        : reddit_1d5d82f2de8b15f113cebf4be29fc7fd.txt
Smallest word count  : 29


In [5]:
from pathlib import Path
from tqdm import tqdm
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

# =====================================================
# PATHS
# =====================================================

BASE = Path(r"E:\AI_Detect")

HUMAN_DIR = BASE / "data/text/human/cleaned"

AI_DIR = BASE / "data/text/ai/cleaned"

META_DIR = BASE / "data/text/ai/metadata"

AI_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================
# MODEL
# =====================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading tokenizer...")

tok = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Loading model...")

dtype = (
    torch.float16
    if torch.cuda.is_available()
    else torch.float32
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto"
)

print("Loaded!")

print("Model:", MODEL_NAME)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            1
        ),
        "GB"
    )

Loading tokenizer...


C:\Users\Shaif\anaconda3\envs\dl\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Shaif\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading model...


Loading weights: 100%|██████████████████████████████████████████████████████████████| 338/338 [00:00<00:00, 357.61it/s]


Loaded!
Model: Qwen/Qwen2.5-1.5B-Instruct
GPU: Quadro P2000
VRAM: 4.0 GB


In [8]:
def generate_doc(text):

    prompt = f"""
Write a new human-like document on the same topic.

Keep the topic.

Do not copy wording.

Source:

{text}
"""

    inputs = tok(

        prompt,

        return_tensors="pt",

        truncation=True,

        max_length=512

    ).to(model.device)

    outputs = model.generate(

        **inputs,

        max_new_tokens=300,

        do_sample=True,

        temperature=0.9,

        top_p=0.95,

        repetition_penalty=1.1,

        pad_token_id=tok.eos_token_id

    )

    return tok.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [7]:
import torch

print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory/1024**3)

Quadro P2000
3.9998779296875


In [ ]:
from tqdm import tqdm
import pandas as pd

files = list(HUMAN_DIR.glob("*.txt"))

metadata = []

# resume support
existing = set()

for fp in AI_DIR.glob("*.txt"):

    existing.add(fp.stem)

print("Already generated:", len(existing))

for fp in tqdm(files):

    try:

        out_name = fp.stem + "_qwen.txt"

        if out_name.replace(".txt", "") in existing:
            continue

        text = fp.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        # ---------------------------------
        # limit input size
        # ---------------------------------

        words = text.split()

        if len(words) < 100:
            continue

        text = " ".join(words[:300])

        ai_text = generate_doc(text)

        # ---------------------------------
        # remove prompt echo
        # ---------------------------------

        ai_text = ai_text.strip()

        # ---------------------------------
        # minimum quality
        # ---------------------------------

        if len(ai_text.split()) < 100:
            continue

        save_path = AI_DIR / out_name

        save_path.write_text(
            ai_text,
            encoding="utf-8"
        )

        metadata.append({

            "human_file": fp.name,

            "ai_file": out_name,

            "model": MODEL_NAME,

            "human_words": len(words),

            "ai_words": len(ai_text.split())

        })

    except Exception as e:

        print("ERROR:", fp.name)
        print(e)

# -----------------------------------------
# metadata
# -----------------------------------------

meta = pd.DataFrame(metadata)

meta_file = META_DIR / "qwen_metadata.csv"

if meta_file.exists():

    old = pd.read_csv(meta_file)

    meta = pd.concat(
        [old, meta],
        ignore_index=True
    )

meta.to_csv(
    meta_file,
    index=False
)

print()

print("Generated:", len(metadata))
print("Metadata:", meta_file)

Already generated: 0


  0%|                                                                             | 5/14514 [01:50<88:23:16, 21.93s/it]